In [3]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

192.168.88.1:50051


RESET

In [14]:
got.mechanical_joint_control(0, 90, 90, 10)    
got.mechanical_clamp_close()
got.mecanum_stop()

STRAIGHT

In [12]:
got.mechanical_joint_control(0, 0, 0, 10)    

CLOSE


In [ ]:
got.mechanical_clamp_close()

OPEN

In [ ]:
got.mechanical_clamp_release()

MANUAL CONTROL

FWD

In [23]:
try: 
    while True:
        got.mecanum_translate_speed(angle=0,speed=100)

except KeyboardInterrupt:
    print("\nStopping...")

    got.mecanum_stop()     

    print("\nCancelled")


Stopping...

Cancelled


LEFT

In [22]:
try: 
    while True:
        got.mecanum_translate_speed(angle=-90,speed=100)

except KeyboardInterrupt:
    print("\nStopping...")

    got.mecanum_stop()     

    print("\nCancelled")


Stopping...

Cancelled


RIGHT

In [20]:
try: 
    while True:
        got.mecanum_translate_speed(angle=90,speed=100)

except KeyboardInterrupt:
    print("\nStopping...")

    got.mecanum_stop()     

    print("\nCancelled")


Stopping...

Cancelled


BACK

In [15]:
try: 
    while True:
        got.mecanum_translate_speed(angle=180,speed=100)

except KeyboardInterrupt:
    print("\nStopping...")

    got.mecanum_stop()     

    print("\nCancelled")


Stopping...

Cancelled


arm holding

In [24]:
import time

got.mechanical_clamp_release() # Open the gripper
time.sleep(2)
got.mechanical_clamp_close() # Close the gripper

got.mechanical_joint_control(0, 30, -20, 1000)

Putdownnicely

In [ ]:
import time

got.mechanical_joint_control(0, -15, -42, 1000)
time.sleep(2)
got.mechanical_clamp_release() # Open the gripper


SPIN RIGHT

In [ ]:
try: 
    while True:
        got.mecanum_turn_speed(turn=3,speed=20)

except KeyboardInterrupt:
    print("\nStopping...")

    got.mecanum_stop()     

    print("\nCancelled")

SPIN LEFT

In [19]:
try: 
    while True:
        got.mecanum_turn_speed(turn=2,speed=20)

except KeyboardInterrupt:
    print("\nStopping...")

    got.mecanum_stop()     

    print("\nCancelled")


Stopping...

Cancelled


Camera


In [ ]:
import cv2
import numpy as np
import time

try:
    from ugot import ugot
except:
    ugot = None


# ---------------------------
# INIT ROBOT
# ---------------------------
def init_robot(ip="192.168.88.1"):
    if ugot is None:
        print("❌ UGOT SDK not installed")
        return None

    try:
        robot = ugot.UGOT()
        robot.initialize(ip)

        try:
            robot.open_camera()
        except:
            pass

        print("✅ Robot connected + camera opened")
        return robot

    except Exception as e:
        print("❌ Failed:", e)
        return None


# ---------------------------
# CAMERA STREAM LOOP
# ---------------------------
def show_robot_camera(robot):
    print("📡 Starting camera stream... Press Q to exit")

    while True:
        try:
            frame = None

            # different SDK versions use different methods
            if hasattr(robot, "get_camera_frame"):
                frame = robot.get_camera_frame()

            elif hasattr(robot, "read_camera_data"):
                raw = robot.read_camera_data()
                if raw is not None:
                    nparr = np.frombuffer(raw, np.uint8)
                    frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

            if frame is None:
                continue

            cv2.imshow("UGOT Robot Camera", frame)

        except Exception as e:
            print("Camera error:", e)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cv2.destroyAllWindows()


# ---------------------------
# RUN
# ---------------------------
if __name__ == "__main__":
    robot = init_robot("192.168.88.1")

    if robot is not None:
        show_robot_camera(robot)

Pose

In [ ]:
import cv2

import numpy as np

import time

from ultralytics import YOLO



try:

    from ugot import ugot

except:

    ugot = None





# ---------------------------

# KEYPOINTS

# ---------------------------

COCO_KEYPOINTS = [

    "nose", "left_eye", "right_eye", "left_ear", "right_ear",

    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",

    "left_wrist", "right_wrist", "left_hip", "right_hip",

    "left_knee", "right_knee", "left_ankle", "right_ankle"

]





# ---------------------------

# INIT ROBOT (SAFE)

# ---------------------------

def init_robot(ip="192.168.88.1"):

    if ugot is None:

        print("❌ UGOT not installed")

        return None



    try:

        robot = ugot.UGOT()

        robot.initialize(ip)



        try:

            robot.load_models([])

        except:

            pass



        try:

            robot.open_camera()

        except:

            pass



        print("✅ Robot object created (camera may still depend on SDK/service)")

        return robot



    except Exception as e:

        print("❌ Robot init failed:", e)

        return None





# ---------------------------

# ROBOT CAMERA (ROBUST + SAFE)

# ---------------------------

def get_robot_frame(robot):

    if robot is None:

        return None



    frame = None



    try:

        if hasattr(robot, "get_camera_frame"):

            frame = robot.get_camera_frame()



        elif hasattr(robot, "read_camera_data"):

            raw = robot.read_camera_data()

            if raw is not None:

                nparr = np.frombuffer(raw, np.uint8)

                frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)



        elif hasattr(robot, "camera"):

            raw = robot.camera.read()

            if raw is not None:

                frame = raw



        if frame is None:

            return None



        frame = cv2.resize(frame, (660, 480))

        return frame



    except:

        return None





# ---------------------------

# SELECT MAIN PERSON

# ---------------------------

def select_main_person(results):

    if not results or results[0].keypoints is None:

        return None



    data = results[0].keypoints.data

    if data is None or len(data) == 0:

        return None



    best_i = 0

    best_score = -1



    for i in range(len(data)):

        person = data[i].detach().cpu().numpy()



        xy = person[:, :2]

        conf = person[:, 2]



        valid = xy[conf > 0.3]

        if len(valid) < 5:

            continue



        score = np.linalg.norm(valid.max(axis=0) - valid.min(axis=0))



        if score > best_score:

            best_score = score

            best_i = i



    return data[best_i].detach().cpu().numpy()





# ---------------------------

# CLASSIFIER

# ---------------------------

def classify_pose(kp, min_conf=0.3):



    def get(name):

        if name not in kp:

            return None

        x, y, c = kp[name]

        if c < min_conf:

            return None

        return np.array([x, y], dtype=np.float32)



    ls = get("left_shoulder")

    rs = get("right_shoulder")

    lw = get("left_wrist")

    rw = get("right_wrist")



    if ls is None or rs is None or lw is None or rw is None:

        return "NONE"



    torso = np.linalg.norm(ls - rs)

    if torso < 1e-6:

        return "NONE"



    up = 0.25 * torso

    down = 0.25 * torso



    left_up = lw[1] < ls[1] - up

    right_up = rw[1] < rs[1] - up



    left_down = lw[1] > ls[1] + down

    right_down = rw[1] > rs[1] + down



    left_mid = not left_up and not left_down

    right_mid = not right_up and not right_down



    dist = abs(lw[0] - rw[0])



    if left_mid and right_mid and dist < 0.55 * torso:

        return "EXIT"



    if left_mid and right_mid and dist > 1.45 * torso:

        return "PICKUP"



    if left_up and right_up:

        return "FORWARD"

    elif left_down and right_down:

        return "BACKWARD"

    elif left_up:

        return "LEFT"

    elif right_up:

        return "RIGHT"



    return "NONE"





# ---------------------------

# DOT SKELETON ONLY

# ---------------------------

def draw_skeleton(frame, kps):

    for x, y, c in kps:

        if c > 0.3:

            cv2.circle(frame, (int(x), int(y)), 4, (0, 255, 0), -1)





# ---------------------------

# UI

# ---------------------------

def draw_ui(frame, state, robot_frame):



    h, w = frame.shape[:2]



    cv2.rectangle(frame, (10, 10), (450, 170), (0, 0, 0), -1)



    cv2.putText(frame, f"STATE: {state}", (20, 40),

                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)



    cv2.putText(frame, "UP = FORWARD / LEFT / RIGHT", (20, 70),

                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)



    cv2.putText(frame, "DOWN = BACKWARD", (20, 95),

                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)



    cv2.putText(frame, "WIDE = PICKUP", (20, 120),

                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)



    cv2.putText(frame, "TOGETHER = EXIT", (20, 145),

                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)



    col = lambda x: (0, 255, 0) if x else (80, 80, 80)



    cv2.circle(frame, (80, h - 60), 6, col(state == "FORWARD"), -1)

    cv2.circle(frame, (40, h - 40), 6, col(state == "LEFT"), -1)

    cv2.circle(frame, (120, h - 40), 6, col(state == "RIGHT"), -1)

    cv2.circle(frame, (80, h - 20), 6, col(state == "BACKWARD"), -1)



    if robot_frame is not None:

        rf = robot_frame

        rh, rw = rf.shape[:2]



        max_w = int(w * 0.5)

        max_h = int(h * 0.5)



        scale = min(max_w / rw, max_h / rh, 1.0)



        rf = cv2.resize(rf, (int(rw * scale), int(rh * scale)))



        rh, rw = rf.shape[:2]



        x1 = w - rw - 10

        y1 = 10



        frame[y1:y1+rh, x1:x1+rw] = rf





# ---------------------------

# MAIN LOOP

# ---------------------------

def run_pose_control(robot=None):



    model = YOLO("yolov8n-pose.pt")



    cap = cv2.VideoCapture(0)

    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)



    history = []

    prev = "NONE"



    PICKUP_COOLDOWN = 3.0

    last_pickup = 0



    # FIXED STATE MACHINE

    pickup_state = 0

    pickup_time = 0



    print("🚀 SYSTEM STARTED")



    while True:

        ret, frame = cap.read()

        if not ret:

            break



        results = model.predict(frame, verbose=False, imgsz=320, conf=0.35)



        person = select_main_person(results)



        cmd = "NONE"



        if person is not None:

            kps = person

            kp_dict = {COCO_KEYPOINTS[i]: kps[i] for i in range(len(COCO_KEYPOINTS))}

            cmd = classify_pose(kp_dict)



            draw_skeleton(frame, kps)



        history.append(cmd)

        if len(history) > 6:

            history.pop(0)



        state = max(set(history), key=history.count)



        robot_frame = get_robot_frame(robot)



        draw_ui(frame, state, robot_frame)



        cv2.imshow("UGOT CONTROL", frame)



        # ---------------------------

        # ROBOT CONTROL

        # ---------------------------

        if robot is not None:

            try:

                now = time.time()



                if state == "FORWARD":

                    robot.mecanum_move_speed(0, 20)



                elif state == "BACKWARD":

                    robot.mecanum_move_speed(1, 20)



                elif state == "LEFT":

                    robot.mecanum_turn_speed(2, 30)



                elif state == "RIGHT":

                    robot.mecanum_turn_speed(3, 30)



                # ---------------------------

                # CLEAN PICKUP SEQUENCE FIXED

                # ---------------------------

                elif state == "PICKUP":

                    if pickup_state == 0 and now - last_pickup > PICKUP_COOLDOWN:

                        robot.mechanical_clamp_release()

                        robot.mechanical_joint_control(angle1=0, angle2=5, angle3=-75, duration=500)

                        time.sleep(1)



                        pickup_state = 1

                        pickup_time = now

                        last_pickup = now



                # STEP 2: WAIT 1 SEC THEN CLOSE

                if pickup_state == 1 and now - pickup_time > 1.0:

                    robot.mechanical_clamp_close()

                    pickup_state = 2

                    pickup_time = now



                # STEP 3: WAIT THEN RESET ARM

                if pickup_state == 2 and now - pickup_time > 1.0:

                    robot.mechanical_joint_control(angle1=0, angle2=60, angle3=-50, duration=1000)

                    pickup_state = 0



                if state not in ["FORWARD", "BACKWARD", "LEFT", "RIGHT", "PICKUP"]:

                    robot.mecanum_stop()



            except Exception as e:

                print("⚠️ Robot error:", e)



        if state == "EXIT":

            break



        if cv2.waitKey(1) & 0xFF == 27:

            break



    cap.release()

    cv2.destroyAllWindows()



    if robot is not None:

        robot.mecanum_stop()





if __name__ == "__main__":

    robot = init_robot("192.168.88.1")

    run_pose_control(robot)

Face

In [ ]:
import cv2
import numpy as np
import face_recognition
import time
import os
import warnings
from collections import Counter

warnings.filterwarnings("ignore", category=UserWarning)

try:
    from ugot import ugot
except ImportError:
    ugot = None

# ===========================================================================
# SETTINGS — tune these if needed
# ===========================================================================
ROBOT_IP          = "192.168.88.1"
FACE_TOLERANCE    = 0.45   # Distance threshold. Lower = stricter. 0.45 is strict but fair.
CONFIRM_THRESHOLD = 2      # How many consecutive matches before we act (anti-flicker)
PROCESS_WIDTH     = 640    # Resize frame to this width before detection (speed + accuracy balance)
SKIP_FRAMES       = 2      # Run detection every N frames; draw cached boxes in between

try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()

# ===========================================================================
# DATABASE — add more photos per person for better accuracy.
# Even just 2-3 photos (front, slight angle, different lighting) helps a lot.
# ===========================================================================
DATABASE_FILES = {
    "ARJUN":  ["Arjun.jpg",  "Arjun_right.jpg", "Arjun_left.jpg",  "Arjun_dark.jpg", "Arjun_far.jpg", "Arjun_light.jpg", "Arjun_robot.jpg", "Arjun_farright.jpg", "Arjun_farleft.jpg"],
    "ANANYA": ["Ananya.jpg", "Ananya_right.jpg", "Ananya_left.jpg", "Ananya_dark.jpg", "Ananya_far.jpg", "Ananya_light.jpg", "Ananya_robot.jpg", "Ananya_farright.jpg", "Ananya_farleft.jpg"],
    "COLEY":  ["Coley.jpg",  "Coley_right.jpg", "Coley_left.jpg", "Coley_dark.jpg", "Coley_far.jpg", "Coley_light.jpg", "Coley_robot.jpg", "Coley_farright.jpg", "Coley_farleft.jpg"],
    "RYAN":   ["Ryan.jpg",   "Ryan_right.jpg", "Ryan_left.jpg", "Ryan_dark.jpg", "Ryan_far.jpg", "Ryan_light.jpg", "Ryan_robot.jpg", "Ryan_farright.jpg", "Ryan_farleft.jpg"],
}


# ===========================================================================
# FIX 1 — SMARTER PREPROCESSING
#
# Previous version ran heavy denoising (fastNlMeansDenoising) on EVERY frame.
# That alone can cost 80-200ms per frame = the main lag source.
# New approach: lightweight CLAHE only. Fixes lighting without the cost.
# ===========================================================================
def preprocess_frame(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    return enhanced


# ===========================================================================
# FIX 2 — RESOLUTION-AWARE DETECTION PIPELINE
#
# The #1 accuracy killer: running HOG on a full-res frame means distant faces
# are too small to detect. But upsampling a big frame is slow.
# Solution: resize frame to a fixed working width (640px) FIRST, then
# upsample once inside face_locations. This gives HOG a consistently-sized
# face regardless of the camera's native resolution.
#
# Also: we compute face_encodings with the known_face_locations parameter
# which reuses the already-detected bounding boxes instead of re-detecting.
# This cuts encoding time roughly in half.
# ===========================================================================
def detect_and_encode(rgb_small):
    locs = face_recognition.face_locations(
        rgb_small,
        number_of_times_to_upsample=1,  # 1 upsample on a pre-resized frame = equivalent to 2 on full res, but faster
        model="hog"
    )
    if not locs:
        return [], []
    encs = face_recognition.face_encodings(
        rgb_small,
        known_face_locations=locs,
        num_jitters=1,          # 1 = fast. 3-5 = more accurate but slower. Keep at 1 for real-time.
        model="large"           # "large" model is significantly more accurate than "small" for recognition
    )
    return locs, encs


# ===========================================================================
# FIX 3 — WEIGHTED VOTING RECOGNITION
#
# Old approach: trust the single closest match. This breaks when the face
# is partially occluded or at an angle — one bad frame causes a wrong ID.
#
# New approach: compare the incoming encoding against ALL known encodings,
# collect their distances, and pick the person whose encodings are
# consistently close (not just the single closest). This naturally handles
# multiple reference photos per person.
# ===========================================================================
def identify_face(enc, known_encs, known_names):
    if not known_encs:
        return "UNKNOWN", 1.0

    distances = face_recognition.face_distance(known_encs, enc)

    # Group distances by person name and take their minimum (best match per person)
    person_best = {}
    for name, d in zip(known_names, distances):
        if name not in person_best or d < person_best[name]:
            person_best[name] = d

    best_person = min(person_best, key=person_best.get)
    best_dist   = person_best[best_person]

    if best_dist < FACE_TOLERANCE:
        return best_person, best_dist
    return "UNKNOWN", best_dist


# ===========================================================================
# FIX 4 — PER-FACE CONFIRMATION COUNTER
#
# Replaces the single global RecognitionBuffer from v2 which mixed
# results from different faces into one buffer.
# Now each tracked face position has its own small deque of recent labels.
# A name is only reported as confirmed once it appears CONFIRM_THRESHOLD
# times in a row. This stops the robot from reacting to a 1-frame flicker.
# ===========================================================================
class FaceConfirmer:
    def __init__(self, threshold=CONFIRM_THRESHOLD):
        self.threshold  = threshold
        self.streak     = {}   # box_idx -> (current_name, streak_count)
        self.confirmed  = {}   # box_idx -> confirmed_name

    def update(self, box_idx, name):
        prev_name, count = self.streak.get(box_idx, (None, 0))
        if name == prev_name:
            count += 1
        else:
            count = 1
        self.streak[box_idx] = (name, count)

        if count >= self.threshold and name != "UNKNOWN":
            self.confirmed[box_idx] = name

        return self.confirmed.get(box_idx, "UNKNOWN")

    def clear_stale(self, active_indices):
        for k in list(self.streak.keys()):
            if k not in active_indices:
                del self.streak[k]
                self.confirmed.pop(k, None)


# ===========================================================================
# LOAD DATABASE
# ===========================================================================
def load_database():
    known_encodings = []
    known_names     = []

    for person_name, filenames in DATABASE_FILES.items():
        loaded = 0
        for filename in filenames:
            img_path = os.path.join(SCRIPT_DIR, filename)
            if not os.path.exists(img_path):
                print(f"    ❌ File not found: {filename}")
                continue
            img = face_recognition.load_image_file(img_path)

            # Resize reference photo to match our working resolution
            h, w = img.shape[:2]
            if w > PROCESS_WIDTH:
                scale = PROCESS_WIDTH / w
                img   = cv2.resize(img, (PROCESS_WIDTH, int(h * scale)))

            locs = face_recognition.face_locations(img, number_of_times_to_upsample=1)
            encs = face_recognition.face_encodings(img, locs, num_jitters=3, model="large")
            # num_jitters=3 at load time (not real-time) = more accurate reference encoding

            if encs:
                known_encodings.append(encs[0])
                known_names.append(person_name)
                loaded += 1
            else:
                print(f"    ⚠️  No face detected: {filename}")

        status = f"✅ {loaded} photo(s)" if loaded else "⚠️  NO PHOTOS FOUND"
        print(f"  {person_name}: {status}")

    return known_encodings, known_names


# ===========================================================================
# ROBOT HELPERS
# ===========================================================================
def get_frame_from_robot(robot):
    try:
        if hasattr(robot, "get_video_frame"):
            return robot.get_video_frame()
        elif hasattr(robot, "get_camera_frame"):
            return robot.get_camera_frame()
        elif hasattr(robot, "read_camera_data"):
            raw = robot.read_camera_data()
            if raw:
                nparr = np.frombuffer(raw, np.uint8)
                return cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    except:
        pass
    return None


def init_robot():
    if ugot is None:
        return None
    try:
        robot = ugot.UGOT()
        robot.initialize(ROBOT_IP)
        robot.load_models(["face_recognition", "word_recognition"])
        robot.open_camera()
        return robot
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return None


def drop_apriltag(robot):
    print("\n[ACTION] Target reached — preparing drop...")
    robot.mecanum_stop()
    robot.mecanum_move_speed_times(direction=1, speed=15, times=10, unit=1)
    time.sleep(0.5)
    robot.mechanical_joint_control(0, -15, -42, 1000)
    time.sleep(1.5)
    robot.mechanical_clamp_release()
    time.sleep(1)
    print("[ACTION] Mission complete. Backing away.")
    robot.mecanum_move_speed_times(direction=1, speed=20, times=12, unit=1)
    robot.mechanical_joint_control(angle1=0, angle2=40, angle3=40, duration=800)


# ===========================================================================
# MAIN LOOP
# ===========================================================================
def run_tracking():
    print("🔄 Loading face database...")
    known_encs, known_names = load_database()
    if not known_encs:
        print("❌ No face images loaded — add photos to the script directory.")
        return

    all_people = sorted(set(known_names))
    print(f"\nLoaded: {all_people}")
    target_input = input("Who should I find? ").strip().upper()
    if target_input not in all_people:
        print(f"⚠️  '{target_input}' not in database. Continuing anyway...")

    robot = init_robot()
    if robot is None:
        return

    confirmer         = FaceConfirmer(threshold=CONFIRM_THRESHOLD)
    is_mission_done   = False
    has_locked_target = False   # True once we've confirmed target at least once

    frame_count    = 0
    cached_boxes   = []   # (l, t, r, b) in ORIGINAL frame coordinates
    cached_labels  = {}   # box_idx -> stable name

    # Zigzag search state (used only after losing a locked target)
    zigzag_step        = 0
    zigzag_frames_done = 0
    ZIGZAG_FRAMES      = 20   # frames to turn in each direction before switching

    print(f"\n🚀 HUNTING: {target_input}")
    print(f"   tolerance={FACE_TOLERANCE} | confirm_streak={CONFIRM_THRESHOLD} | skip={SKIP_FRAMES} frames\n")

    while not is_mission_done:
        frame = get_frame_from_robot(robot)
        if frame is None:
            continue

        frame_count += 1
        orig_h, orig_w = frame.shape[:2]

        # ---------------------------------------------------------------
        # RESIZE to working width — keeps detection speed consistent
        # regardless of camera resolution
        # ---------------------------------------------------------------
        scale     = PROCESS_WIDTH / orig_w
        small     = cv2.resize(frame, (PROCESS_WIDTH, int(orig_h * scale)))
        processed = preprocess_frame(small)
        rgb_small = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)

        # ---------------------------------------------------------------
        # DETECTION — only every SKIP_FRAMES frames
        # ---------------------------------------------------------------
        if frame_count % SKIP_FRAMES == 0:
            locs, encs = detect_and_encode(rgb_small)

            new_boxes  = []
            new_labels = {}

            active_indices = set()

            for idx, ((top, right, bottom, left), enc) in enumerate(zip(locs, encs)):
                active_indices.add(idx)

                # Scale bounding box back to original frame coordinates
                l_orig = int(left   / scale)
                t_orig = int(top    / scale)
                r_orig = int(right  / scale)
                b_orig = int(bottom / scale)
                new_boxes.append((l_orig, t_orig, r_orig, b_orig))

                # Identify & confirm
                raw_name, dist_val = identify_face(enc, known_encs, known_names)
                stable_name = confirmer.update(idx, raw_name)
                new_labels[idx] = (stable_name, dist_val)

            confirmer.clear_stale(active_indices)
            cached_boxes  = new_boxes
            cached_labels = new_labels

        # ---------------------------------------------------------------
        # DRAW & TRACK TARGET
        # ---------------------------------------------------------------
        found_target = False
        target_box   = None

        for idx, (l, t, r, b) in enumerate(cached_boxes):
            stable_name, dist_val = cached_labels.get(idx, ("UNKNOWN", 1.0))
            is_target = (stable_name == target_input)

            if is_target:
                found_target = True
                target_box   = (l, t, r, b)
                color        = (0, 255, 0)
            elif stable_name == "UNKNOWN":
                color = (100, 100, 100)
            else:
                color = (0, 80, 255)

            cv2.rectangle(frame, (l, t), (r, b), color, 2)

            # Show name + confidence distance
            label = f"{stable_name} ({dist_val:.2f})"
            cv2.putText(frame, label, (l, t - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        # ---------------------------------------------------------------
        # OCR fallback
        # ---------------------------------------------------------------
        found_by_text = False
        try:
            res = robot.get_word_recognition_result()
            if res:
                ocr_words = [str(w).upper() for w in res]
                found_by_text = any(target_input in w for w in ocr_words)
                cv2.putText(frame, f"OCR: {ocr_words}", (10, 55),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 0), 1)
        except:
            pass

        # ---------------------------------------------------------------
        # MOVEMENT LOGIC
        # ---------------------------------------------------------------
        if found_target or found_by_text:
            has_locked_target = True
            # Reset zigzag state whenever target is visible
            zigzag_step        = 0
            zigzag_frames_done = 0
            robot.mecanum_stop()

            if target_box:
                l, t, r, b = target_box
                cx = (l + r) // 2
                fw = r - l          # face pixel width = proxy for distance
            else:
                cx = orig_w // 2
                fw = 60

            err = cx - (orig_w // 2)

            if abs(err) > 40:
                # Turn to centre the face
                direction = 2 if err < 0 else 3
                robot.mecanum_turn_speed(direction, 12)
                cv2.putText(frame, f"CENTERING err={err}", (10, 80),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

            elif fw < 70:
                # Approach — face is too small = still far away
                # (raised from 130 → 90 so robot stops further back before drop)
                print(f"[MOVE] Approaching {target_input}... face_w={fw}px")
                robot.mecanum_move_speed(0, 18)

            else:
                # Close enough — execute drop
                print(f"🎯 LOCKED & POSITIONED — face_w={fw}px")
                drop_apriltag(robot)
                is_mission_done = True

        else:
            if not has_locked_target:
                # Haven't found target yet — keep spinning to search
                robot.mecanum_turn_speed(2, 10)
            else:
                # Had target, briefly lost it — zigzag left/right to relocate
                zigzag_frames_done += 1
                if zigzag_frames_done >= ZIGZAG_FRAMES:
                    zigzag_frames_done = 0
                    zigzag_step += 1   # alternate direction each step

                direction = 2 if (zigzag_step % 2 == 0) else 3  # 2=left, 3=right
                robot.mecanum_turn_speed(direction, 10)

                cv2.putText(frame, f"ZIGZAG SEARCHING... step={zigzag_step}", (10, 80),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 1)
                print(f"[ZIGZAG] Lost {target_input}, searching step={zigzag_step} dir={'LEFT' if direction==2 else 'RIGHT'}")

        # ---------------------------------------------------------------
        # HUD
        # ---------------------------------------------------------------
        hud = f"TARGET: {target_input} | {'FOUND ✓' if found_target else 'SEARCHING...'}"
        cv2.putText(frame, hud, (10, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255) if found_target else (0, 165, 255), 2)

        cv2.imshow("Robot Tracking V3", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            print("ESC pressed — stopping.")
            break

    robot.mecanum_stop()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    run_tracking()

TEST Part 3 Yellow stopper on

In [ ]:
# ---------------------------
# SETTINGS & CONSTANTS
# ---------------------------
COCO_KEYPOINTS = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

# ── Face recognition settings (from V3) ──────────────────────────────
CONFIRM_THRESHOLD = 2
SKIP_FRAMES       = 2

# ── Movement speeds ───────────────────────────────────────────────────
SPIN_FAST     = 23
SPIN_SLOW     = 9
APPROACH_FAST = 18
APPROACH_SLOW = 13
CENTER_SPEED  = 12

# ── Colour mark (Box A) stop settings ────────────────────────────────
# Detects ANY strong colour patch — works regardless of mark colour.
# The robot camera looks down; when the coloured diamond fills enough
# of the frame the robot stops, then nudges forward to centre on it.
COLORMARK_PIXEL_THRESHOLD = 0.04    # 4% of frame must be saturated colour
COLORMARK_CONFIRM_FRAMES  = 2       # consecutive matching frames needed
COLORMARK_CREEP_SPEED     = 12      # slow forward speed while searching
COLORMARK_NUDGE_SPEED     = 20      # speed for the final centering nudge
COLORMARK_NUDGE_TIME      = 2.1    # seconds to nudge forward after detection


# ---------------------------
# COLOUR MARK DETECTION (Box A) — any colour
# ---------------------------
def _check_colormark(frame):
    """
    Returns True when the frame contains enough saturated colour pixels
    to indicate the robot is over the coloured mark.
    Works for any mark colour (yellow, red, blue, green, etc.)
    by checking saturation rather than a specific hue.
    """
    try:
        hsv        = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        total      = hsv.shape[0] * hsv.shape[1]
        # High saturation (>100) + reasonable brightness (>60) = coloured mark
        lower      = np.array([0,   100, 60],  dtype=np.uint8)
        upper      = np.array([179, 255, 255], dtype=np.uint8)
        mask       = cv2.inRange(hsv, lower, upper)
        matched    = int(cv2.countNonZero(mask))
        return (matched / total) >= COLORMARK_PIXEL_THRESHOLD
    except Exception:
        return False


def stop_on_colormark(robot, timeout=10.0):
    """
    1. Creep forward until the coloured mark is detected for
       COLORMARK_CONFIRM_FRAMES consecutive frames.
    2. Stop immediately.
    3. Nudge forward a fixed amount (COLORMARK_NUDGE_TIME seconds)
       so the robot is centred on top of the mark.
    Returns True if mark was found, False if timed out.
    """
    print("[BOX A] Driving to colour mark...")
    confirm_streak = 0
    start = time.time()

    while time.time() - start < timeout:
        # Creep forward slowly
        safe_call(robot.mecanum_move_speed, 0, COLORMARK_CREEP_SPEED)

        # Check robot camera for colour mark
        frame = get_robot_frame(robot)
        if frame is not None:
            if _check_colormark(frame):
                confirm_streak += 1
                print(f"[BOX A] Colour mark detected — streak {confirm_streak}/{COLORMARK_CONFIRM_FRAMES}")
            else:
                confirm_streak = 0

            if confirm_streak >= COLORMARK_CONFIRM_FRAMES:
                # Stop as soon as mark is confirmed
                safe_call(robot.mecanum_stop)
                time.sleep(0.1)

                # ── Hardcoded forward nudge to centre on the mark ──────
                print(f"[BOX A] Nudging forward {COLORMARK_NUDGE_TIME}s to centre on mark...")
                safe_call(robot.mecanum_move_speed, 0, COLORMARK_NUDGE_SPEED)
                time.sleep(COLORMARK_NUDGE_TIME)
                safe_call(robot.mecanum_stop)
                # ───────────────────────────────────────────────────────

                print("[BOX A] ✅ Stopped ON colour mark in Box A")
                return True

        time.sleep(0.05)

    safe_call(robot.mecanum_stop)
    print("[BOX A] ⚠️  Colour mark not found — timed out, stopping anyway")
    return False


# ---------------------------
# ROBOT UTILITIES
# ---------------------------
def get_robot_frame(robot):
    """Return a 660×480 BGR frame, or None. Never raises."""
    if robot is None:
        return None
    try:
        if hasattr(robot, "get_camera_frame"):
            frame = robot.get_camera_frame()
        elif hasattr(robot, "read_camera_data"):
            raw = robot.read_camera_data()
            if raw is None:
                return None
            frame = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)
        elif hasattr(robot, "camera"):
            frame = robot.camera.read()
        else:
            return None
        if frame is not None:
            frame = cv2.resize(frame, (660, 480))
        return frame
    except Exception:
        return None


def safe_call(fn, *args, **kwargs):
    """Call any robot method without crashing the program."""
    try:
        fn(*args, **kwargs)
    except Exception as e:
        print(f"  [robot error in {fn.__name__}]: {e}")


# ---------------------------
# POSE CONTROL HELPERS
# ---------------------------
def select_main_person(results):
    if not results or results[0].keypoints is None:
        return None
    data = results[0].keypoints.data
    if data is None or len(data) == 0:
        return None
    best_i, best_score = 0, -1
    for i in range(len(data)):
        person = data[i].detach().cpu().numpy()
        xy, conf = person[:, :2], person[:, 2]
        valid = xy[conf > 0.3]
        if len(valid) < 5:
            continue
        score = float(np.linalg.norm(valid.max(axis=0) - valid.min(axis=0)))
        if score > best_score:
            best_score, best_i = score, i
    return data[best_i].detach().cpu().numpy() if best_score > -1 else None


def classify_pose(kp, min_conf=0.3):
    def get(name):
        if name not in kp:
            return None
        x, y, c = kp[name]
        return np.array([x, y], dtype=np.float32) if c >= min_conf else None

    ls, rs = get("left_shoulder"), get("right_shoulder")
    lw, rw = get("left_wrist"),    get("right_wrist")
    if any(v is None for v in (ls, rs, lw, rw)):
        return "NONE"
    torso = float(np.linalg.norm(ls - rs))
    if torso < 1e-6:
        return "NONE"
    thr        = 0.25 * torso
    left_up    = lw[1] < ls[1] - thr
    right_up   = rw[1] < rs[1] - thr
    left_down  = lw[1] > ls[1] + thr
    right_down = rw[1] > rs[1] + thr
    left_mid   = not left_up  and not left_down
    right_mid  = not right_up and not right_down
    dist       = abs(lw[0] - rw[0])
    if left_mid  and right_mid  and dist < 0.55 * torso: return "EXIT"
    if left_up   and right_up:                            return "FORWARD"
    if left_down and right_down:                          return "BACKWARD"
    if left_up:                                           return "LEFT"
    if right_up:                                          return "RIGHT"
    return "NONE"


def draw_skeleton(frame, kps):
    for x, y, c in kps:
        if c > 0.3:
            cv2.circle(frame, (int(x), int(y)), 4, (0, 255, 0), -1)


def draw_ui(frame, state, robot_frame):
    h, w = frame.shape[:2]
    cv2.rectangle(frame, (10, 10), (450, 170), (0, 0, 0), -1)
    cv2.putText(frame, f"STATE: {state}",             (20,  40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    cv2.putText(frame, "UP = FORWARD / LEFT / RIGHT", (20,  70), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0),   1)
    cv2.putText(frame, "DOWN = BACKWARD",             (20,  95), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255),   1)
    cv2.putText(frame, "TOGETHER = EXIT to Stage 2",  (20, 145), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    if robot_frame is not None:
        rh, rw_px = robot_frame.shape[:2]
        scale = min((w * 0.5) / rw_px, (h * 0.5) / rh, 1.0)
        rf = cv2.resize(robot_frame, (int(rw_px * scale), int(rh * scale)))
        rh, rw_px = rf.shape[:2]
        frame[10:10+rh, w-rw_px-10:w-10] = rf


# ---------------------------
# FACE DETECTION PIPELINE (V3)
# ---------------------------
def preprocess_frame(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    return enhanced


def detect_and_encode(rgb_small):
    locs = face_recognition.face_locations(
        rgb_small,
        number_of_times_to_upsample=1,
        model="hog"
    )
    if not locs:
        return [], []
    encs = face_recognition.face_encodings(
        rgb_small,
        known_face_locations=locs,
        num_jitters=1,
        model="large"
    )
    return locs, encs


def identify_face(enc, known_encs, known_names):
    if not known_encs:
        return "UNKNOWN", 1.0

    distances = face_recognition.face_distance(known_encs, enc)

    person_best = {}
    for name, d in zip(known_names, distances):
        if name not in person_best or d < person_best[name]:
            person_best[name] = d

    best_person = min(person_best, key=person_best.get)
    best_dist   = person_best[best_person]

    if best_dist < FACE_TOLERANCE:
        return best_person, best_dist
    return "UNKNOWN", best_dist


class FaceConfirmer:
    def __init__(self, threshold=CONFIRM_THRESHOLD):
        self.threshold  = threshold
        self.streak     = {}
        self.confirmed  = {}

    def update(self, box_idx, name):
        prev_name, count = self.streak.get(box_idx, (None, 0))
        if name == prev_name:
            count += 1
        else:
            count = 1
        self.streak[box_idx] = (name, count)

        if count >= self.threshold and name != "UNKNOWN":
            self.confirmed[box_idx] = name

        return self.confirmed.get(box_idx, "UNKNOWN")

    def clear_stale(self, active_indices):
        for k in list(self.streak.keys()):
            if k not in active_indices:
                del self.streak[k]
                self.confirmed.pop(k, None)


# ---------------------------
# DROP ACTION (V3)
# ---------------------------
def drop_apriltag(robot):
    print("\n[ACTION] Target reached — preparing drop...")
    safe_call(robot.mecanum_stop)
    safe_call(robot.mecanum_move_speed_times, direction=1, speed=5, times=2, unit=1)
    time.sleep(0.5)
    safe_call(robot.mechanical_joint_control, angle1=0, angle2=-15, angle3=-42, duration=1000)
    time.sleep(1.5)
    safe_call(robot.mechanical_clamp_release)
    time.sleep(1)
    print("[ACTION] Mission complete. Backing away.")
    safe_call(robot.mecanum_move_speed_times, direction=1, speed=20, times=12, unit=1)
    safe_call(robot.mechanical_joint_control, angle1=0, angle2=40, angle3=40, duration=800)


# ---------------------------
# STAGE 1 — POSE CONTROL
# ---------------------------
def run_stage1(robot):
    print("🚀 STAGE 1: POSE CONTROL")

    history = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = pose_model.predict(frame, verbose=False, imgsz=320, conf=0.35)
        person  = select_main_person(results)
        cmd = "NONE"
        if person is not None:
            kp_dict = {COCO_KEYPOINTS[i]: person[i] for i in range(len(COCO_KEYPOINTS))}
            cmd = classify_pose(kp_dict)
            draw_skeleton(frame, person)

        history.append(cmd)
        if len(history) > 6:
            history.pop(0)
        state = max(set(history), key=history.count)

        robot_frame = get_robot_frame(robot)
        draw_ui(frame, state, robot_frame)
        cv2.imshow("UGOT CONTROL", frame)

        if robot is not None:
            if   state == "FORWARD":  safe_call(robot.mecanum_move_speed, 0, 20)
            elif state == "BACKWARD": safe_call(robot.mecanum_move_speed, 1, 20)
            elif state == "LEFT":     safe_call(robot.mecanum_turn_speed, 2, 30)
            elif state == "RIGHT":    safe_call(robot.mecanum_turn_speed, 3, 30)
            else:                     safe_call(robot.mecanum_stop)

        if state == "EXIT" or (cv2.waitKey(1) & 0xFF == 27):
            print("[S1] EXIT pose detected — stopping and finding colour mark in Box A...")
            safe_call(robot.mecanum_stop)
            break

    cap.release()
    cv2.destroyAllWindows()
    if robot:
        safe_call(robot.mecanum_stop)
    time.sleep(0.3)

    # ── Automatically drive onto and stop on the colour mark in Box A ──
    stop_on_colormark(robot, timeout=10.0)
    time.sleep(0.5)


# ---------------------------
# STAGE 2 — FACE RECOGNITION (V3)
# ---------------------------
def run_stage2(robot, known_encs, known_names, target_input, spin_code):
    print(f"\n🚀 STAGE 2: Searching for {target_input}")
    print(f"   tolerance={FACE_TOLERANCE} | confirm_streak={CONFIRM_THRESHOLD} | skip={SKIP_FRAMES} frames\n")

    confirmer         = FaceConfirmer(threshold=CONFIRM_THRESHOLD)
    is_mission_done   = False
    has_locked_target = False

    frame_count    = 0
    cached_boxes   = []
    cached_labels  = {}

    # Zigzag search state (used only after losing a locked target)
    zigzag_step        = 0
    zigzag_frames_done = 0
    ZIGZAG_FRAMES      = 20

    # Show placeholder window immediately so waitKey works
    placeholder = np.zeros((480, 660, 3), dtype=np.uint8)
    cv2.putText(placeholder, f"Searching for {target_input}...",
                (60, 240), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    cv2.imshow("Robot Tracking V3", placeholder)
    cv2.waitKey(1)

    while not is_mission_done:
        frame = get_robot_frame(robot)
        if frame is None:
            cv2.imshow("Robot Tracking V3", placeholder)
            if cv2.waitKey(50) & 0xFF == 27:
                break
            continue

        frame_count += 1
        orig_h, orig_w = frame.shape[:2]

        # Resize to working width
        scale     = PROCESS_WIDTH / orig_w
        small     = cv2.resize(frame, (PROCESS_WIDTH, int(orig_h * scale)))
        processed = preprocess_frame(small)
        rgb_small = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)

        # Detection — only every SKIP_FRAMES frames
        if frame_count % SKIP_FRAMES == 0:
            locs, encs = detect_and_encode(rgb_small)

            new_boxes      = []
            new_labels     = {}
            active_indices = set()

            for idx, ((top, right, bottom, left), enc) in enumerate(zip(locs, encs)):
                active_indices.add(idx)

                l_orig = int(left   / scale)
                t_orig = int(top    / scale)
                r_orig = int(right  / scale)
                b_orig = int(bottom / scale)
                new_boxes.append((l_orig, t_orig, r_orig, b_orig))

                raw_name, dist_val = identify_face(enc, known_encs, known_names)
                stable_name = confirmer.update(idx, raw_name)
                new_labels[idx] = (stable_name, dist_val)

            confirmer.clear_stale(active_indices)
            cached_boxes  = new_boxes
            cached_labels = new_labels

        # Draw & find target
        found_target = False
        target_box   = None

        for idx, (l, t, r, b) in enumerate(cached_boxes):
            stable_name, dist_val = cached_labels.get(idx, ("UNKNOWN", 1.0))
            is_target = (stable_name == target_input)

            if is_target:
                found_target = True
                target_box   = (l, t, r, b)
                color        = (0, 255, 0)
            elif stable_name == "UNKNOWN":
                color = (100, 100, 100)
            else:
                color = (0, 80, 255)

            cv2.rectangle(frame, (l, t), (r, b), color, 2)
            label = f"{stable_name} ({dist_val:.2f})"
            cv2.putText(frame, label, (l, t - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        # OCR fallback
        found_by_text = False
        try:
            res = robot.get_word_recognition_result() if robot else None
            if res:
                ocr_words = [str(w).upper() for w in res]
                found_by_text = any(target_input in w for w in ocr_words)
                cv2.putText(frame, f"OCR: {ocr_words}", (10, 55),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 0), 1)
        except Exception:
            pass

        # Movement logic
        if found_target or found_by_text:
            has_locked_target  = True
            zigzag_step        = 0
            zigzag_frames_done = 0
            safe_call(robot.mecanum_stop)

            if target_box:
                l, t, r, b = target_box
                cx = (l + r) // 2
                fw = r - l
            else:
                cx = orig_w // 2
                fw = 60

            err = cx - (orig_w // 2)

            if abs(err) > 40:
                direction = 2 if err < 0 else 3
                safe_call(robot.mecanum_turn_speed, direction, CENTER_SPEED)
                cv2.putText(frame, f"CENTERING err={err}", (10, 80),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

            elif fw < 75:
                print(f"[MOVE] Approaching {target_input}... face_w={fw}px")
                safe_call(robot.mecanum_move_speed, 0, APPROACH_FAST)

            else:
                print(f"🎯 LOCKED & POSITIONED — face_w={fw}px")
                drop_apriltag(robot)
                is_mission_done = True

        else:
            if not has_locked_target:
                # Haven't found target yet — keep spinning to search
                safe_call(robot.mecanum_turn_speed, spin_code, SPIN_FAST)
            else:
                # Had target, briefly lost it — zigzag left/right to relocate
                zigzag_frames_done += 1
                if zigzag_frames_done >= ZIGZAG_FRAMES:
                    zigzag_frames_done = 0
                    zigzag_step += 1

                direction = 2 if (zigzag_step % 2 == 0) else 3
                safe_call(robot.mecanum_turn_speed, direction, SPIN_SLOW)

                cv2.putText(frame, f"ZIGZAG SEARCHING... step={zigzag_step}", (10, 80),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 1)
                print(f"[ZIGZAG] Lost {target_input}, searching step={zigzag_step} dir={'LEFT' if direction==2 else 'RIGHT'}")

        # HUD
        hud = f"TARGET: {target_input} | {'FOUND ✓' if found_target else 'SEARCHING...'}"
        cv2.putText(frame, hud, (10, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255) if found_target else (0, 165, 255), 2)

        cv2.imshow("Robot Tracking V3", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            print("ESC pressed — stopping.")
            break

    safe_call(robot.mecanum_stop)
    cv2.destroyAllWindows()


# ---------------------------
# MAIN
# ---------------------------
def run_system():
    if not known_names:
        print("❌ No faces in database — cannot continue.")
        return

    print(f"\nKnown faces: {known_names}")

    while True:
        target_input = input("Who should I find? ").strip().upper()
        if target_input in known_names:
            break
        print(f"  ⚠️  '{target_input}' not found. Choose from: {known_names}")

    while True:
        spin_dir = input("Spin direction to search (left / right): ").strip().lower()
        if spin_dir in ("left", "right"):
            break
        print("  ⚠️  Please type 'left' or 'right'.")

    spin_code = 2 if spin_dir == "left" else 3
    print(f"\n✅ Ready: find '{target_input}', spin {spin_dir}.")
    print("Starting Stage 1 (pose control)...\n")

    # Stage 1 — pose control, ends by stopping ON colour mark in Box A
    run_stage1(robot)

    # Stage 2 — pass known_encs/known_names so no reload needed
    run_stage2(robot, known_encs, known_names, target_input, spin_code)

    print("\nPart3 finished.")
    if robot:
        safe_call(robot.mecanum_stop)


if __name__ == "__main__":
    run_system()